<a href="https://colab.research.google.com/github/saisathwik2703/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saisathwik2703/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git /content/flyrank-ml-internship-starter

fatal: destination path '/content/flyrank-ml-internship-starter' already exists and is not an empty directory.


## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*


## 1. My lane (or freestyle) and why

**Provisional lane: Lane 4 — CTR / Engagement opportunity scoring.**

I chose this lane because the starter data contains search visibility, position-tier, impressions, and click-through-rate (CTR) measurements that can support a practical review queue. The goal is not just to train a model; it is to identify visible pages that appear to under-capture clicks compared with pages in a similar position tier. This could help a search/content team decide which pages deserve metadata, content, intent-match, engagement, or monitoring review first. I am keeping the lane provisional because the evidence may change after deeper signal and leakage checks.

In [6]:
# This cell is for CODE (numbers, a query, a check).
import pandas as pd

DATA_PATH = "/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print(f"Starter dataset rows: {len(df):,}")
print(f"Starter dataset columns: {df.shape[1]}")


Starter dataset rows: 30,000
Starter dataset columns: 44


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

## 2. The question: decision, action, cost of a wrong call

**Research question:** Which visible pages under-capture clicks relative to the CTR expected for their position tier, and therefore deserve review first?

**Unit of analysis:** A page-level observation in the starter dataset, using the available 90-day search-performance measurements.

**Decision:** Decide which pages should enter a prioritized CTR/engagement review queue.

**Who acts:** A content or SEO reviewer can inspect the highest-priority pages and decide whether to rewrite metadata, improve intent match or snippet structure, improve on-page engagement, or simply monitor the page.

**Output:** A ranked list of pages with an opportunity score, a position-adjusted CTR gap, supporting reason codes, and a confidence/volume check.

**Cost of a wrong call:** A false positive can use limited review time and lead to unnecessary content or metadata changes. A false negative can leave a genuine click opportunity unreviewed. Low-volume pages are especially risky because their CTR can be noisy, so minimum-impression rules will be important.

**Why data/ML can help:** Raw CTR is strongly affected by search visibility, so comparing a page with similar position-tier peers is more useful than simply ranking every page by raw CTR. Data can measure these differences consistently, while a later ML model can combine multiple available signals into a review-priority score. This is therefore a decision-support problem: the useful output is a ranked list that helps someone decide what to review first, not a claim that every low-CTR page needs a rewrite.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Quick framing check

required_columns = ["ctr", "impressions_90d", "position_tier"]

missing = [c for c in required_columns if c not in df.columns]

print("Required fields available:", not missing)

if missing:
    print("Missing columns:", missing)
else:
    print("The starter data contains CTR, impressions, and position-tier fields needed for this question.")

Required fields available: True
The starter data contains CTR, impressions, and position-tier fields needed for this question.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

## 3. Quick look at the data (2–3 real numbers)

The starter dataset contains **30,000 page-level observations**. In an initial check using pages with at least 100 impressions, mean CTR differed substantially by position tier: **page_1 was about 35.48%**, **top_3 was about 33.41%**, while **deep was about 5.54%**. These numbers make Lane 4 worth investigating because raw CTR cannot be interpreted fairly without considering visibility/position. The large difference between position tiers suggests that the next step should compare pages with similar positions rather than simply ranking every page by raw CTR.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Reproduce the supporting numbers directly from the starter CSV.

visible = df[df["impressions_90d"] >= 100].copy()

print(f"Pages in starter dataset: {len(df):,}")

ctr_by_tier = (
    visible.groupby("position_tier")["ctr"]
    .mean()
    .sort_values(ascending=False)
)

print("\nMean CTR by position tier (impressions_90d >= 100):")
print(ctr_by_tier)

for tier in ["page_1", "top_3", "deep"]:
    if tier in ctr_by_tier.index:
        print(
            f"{tier}: "
            f"{ctr_by_tier[tier]:.4f} "
            f"({ctr_by_tier[tier] * 100:.2f}%)"
        )

Pages in starter dataset: 30,000

Mean CTR by position tier (impressions_90d >= 100):
position_tier
page_1      0.354760
top_3       0.334128
striking    0.255782
page_3_5    0.142359
deep        0.055415
Name: ctr, dtype: float64
page_1: 0.3548 (35.48%)
top_3: 0.3341 (33.41%)
deep: 0.0554 (5.54%)


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

## 4. Careful words: what I can and can't claim

### What I can claim

The starter dataset shows an **observed relationship** between search position tier and CTR, and it provides **measured signals** to investigate position-adjusted click opportunity. I can build a **directional, evidence-backed** ranking of pages for review and test whether that ranking is useful under a defined validation setup. Any final recommendation will be framed as **observed, measured, directional, and decision-support**.

### What I cannot claim

I cannot say that a low CTR proves a bad title, meta description, content quality, or search intent. I cannot claim that a particular change caused more clicks unless an appropriate experiment or causal design is used. I cannot claim that these results prove a Google ranking-algorithm factor or predict Google's algorithm. I will also avoid using future information as a feature and will check minimum-volume requirements so that noisy CTR estimates do not dominate the ranking.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Public-safe language check for this notebook.

careful_words = [
    "observed",
    "measured",
    "directional",
    "decision-support"
]

for word in careful_words:
    print(f"✓ {word}")

print(
    "\nNo client names, URLs, private queries, titles, "
    "or keywords are used in this notebook."
)


✓ observed
✓ measured
✓ directional
✓ decision-support

No client names, URLs, private queries, titles, or keywords are used in this notebook.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.